## 1. 라이브러리 임포트

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE

%matplotlib inline
shap.initjs()

## 2. 데이터 로드

In [ ]:
datasets = pd.read_csv('heart.csv')

### 데이터 특성 설명
| 특성 | 설명 |
|---|---|
| Age | 나이 |
| Sex | 성별 (M/F) |
| ChestPainType | 흉통 유형 (TA: 전형적 협심증, ATA: 비전형적, NAP: 비협심증성, ASY: 무증상) |
| RestingBP | 안정 시 혈압 |
| Cholesterol | 콜레스테롤 |
| FastingBS | 공복 혈당 (1: >120mg/dl, 0: ≤120mg/dl) |
| RestingECG | 안정 시 심전도 결과 |
| MaxHR | 최대 심박수 |
| ExerciseAngina | 운동 시 협심증 (Y/N) |
| Oldpeak | 운동 유발 ST 강하 |
| ST_Slope | ST 분절 기울기 |
| HeartDisease | 타겟 (0: 정상, 1: 심장질환) |

## 3. 데이터 확인

In [ ]:
# 결측치 확인
datasets.isnull().sum()

In [ ]:
datasets.head()

In [ ]:
datasets.shape

In [ ]:
datasets['ChestPainType'].unique()

In [ ]:
datasets.info()

## 4. 탐색적 데이터 분석 (EDA)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
fig.suptitle("Feature Distributions", fontsize=18)

sns.histplot(x=datasets["Age"],            ax=axes[0,0]).set_title("Age")
sns.countplot(data=datasets, x="Sex",      ax=axes[0,1]).set_title("Sex")
sns.countplot(x=datasets["ChestPainType"], ax=axes[0,2]).set_title("ChestPainType")
sns.histplot(x=datasets["RestingBP"],      ax=axes[1,0]).set_title("RestingBP")
sns.histplot(x=datasets["Cholesterol"],    ax=axes[1,1]).set_title("Cholesterol")
sns.countplot(x=datasets["FastingBS"],     ax=axes[1,2]).set_title("FastingBS")
sns.countplot(x=datasets["RestingECG"],    ax=axes[2,0]).set_title("RestingECG")
sns.histplot(x=datasets["MaxHR"],          ax=axes[2,1]).set_title("MaxHR")
sns.countplot(x=datasets["ST_Slope"],      ax=axes[2,2]).set_title("ST_Slope")

plt.tight_layout()
plt.show()

## 5. 심장질환 상관 확인

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.countplot(data=datasets, x="ST_Slope", hue="HeartDisease", ax=axes[0]).set_title("ST_Slope vs HeartDisease")
sns.pointplot(data=datasets, x="HeartDisease", y="Cholesterol", ax=axes[1]).set_title("Cholesterol vs HeartDisease")
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(datasets, hue="HeartDisease")
plt.show()

## 6. 데이터 전처리

In [ ]:
# ExerciseAngina: Y/N → True/False
datasets.ExerciseAngina = datasets.ExerciseAngina == "Y"

# Sex: M/F → True/False, 컬럼명 변경
datasets.Sex = datasets.Sex == "M"
datasets.rename(columns={"Sex": "Male"}, inplace=True)

# 범주형 변수 원-핫 인코딩 (ChestPainType, RestingECG, ST_Slope)
datasets = pd.get_dummies(datasets)

In [ ]:
# 타겟과 상관관계 확인
datasets.corr()["HeartDisease"].sort_values(ascending=False)

In [ ]:
# 상관관계 낮은 특성 제거
datasets = datasets.drop(columns=[
    "RestingECG_LVH", "ChestPainType_TA",
    "RestingECG_Normal", "RestingECG_ST", "RestingBP"
])
print("최종 특성 수:", datasets.shape[1])
datasets.head()

## 7. 모델링 (XGBoost)

In [ ]:
X = datasets.drop(columns="HeartDisease")
y = datasets["HeartDisease"]

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = xgb.XGBClassifier(random_state=42, eval_metric="logloss")
model.fit(x_train, y_train)

y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, [round(v) for v in y_pred])
print(f"Test Accuracy (SMOTE 적용 전): {accuracy * 100:.2f}%")

## 8. SMOTE 오버샘플링

In [ ]:
smote = SMOTE(random_state=0)
X_train_over, y_train_over = smote.fit_resample(x_train, y_train)

print(f"SMOTE 적용 전 훈련 데이터: {x_train.shape}")
print(f"SMOTE 적용 후 훈련 데이터: {X_train_over.shape}")
print(f"\nSMOTE 전  - Class 1: {sum(y_train==1)}, Class 0: {sum(y_train==0)}")
print(f"SMOTE 후  - Class 1: {sum(y_train_over==1)}, Class 0: {sum(y_train_over==0)}")

In [ ]:
model.fit(X_train_over, y_train_over)

y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, [round(v) for v in y_pred])
print(f"Test Accuracy (SMOTE 적용 후): {accuracy * 100:.2f}%")

## 9. K-Fold 교차검증 (5-Fold)

In [ ]:
def run_kfold(features, labels, title):
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_accuracy = []
    for fold, (train_idx, val_idx) in enumerate(kfold.split(features), 1):
        X_tr, X_val = features.iloc[train_idx], features.iloc[val_idx]
        y_tr, y_val = labels.iloc[train_idx], labels.iloc[val_idx]
        model.fit(X_tr, y_tr)
        acc = round(accuracy_score(y_val, model.predict(X_val)), 4)
        print(f"  Fold {fold}: {acc:.4f}")
        cv_accuracy.append(acc)
    mean_acc = np.mean(cv_accuracy)
    print(f"  Mean: {mean_acc:.4f}")

    values = [v * 100 for v in cv_accuracy] + [mean_acc * 100]
    colors = ["dodgerblue"] * 5 + ["C2"]
    plt.figure(figsize=(8, 5))
    plt.bar(range(6), values, color=colors)
    plt.xticks(range(6), ["1","2","3","4","5","mean"])
    plt.ylabel("Accuracy (%)")
    plt.ylim(70, 100)
    plt.title(title)
    plt.show()

print("[Before SMOTE]")
run_kfold(x_train, y_train, "K-Fold Cross Validation (Before SMOTE)")

model.fit(X_train_over, y_train_over)

print("\n[After SMOTE]")
run_kfold(X_train_over, y_train_over, "K-Fold Cross Validation (After SMOTE)")

# SHAP 분석을 위해 전체 SMOTE 훈련 데이터로 재학습
model.fit(X_train_over, y_train_over)

## 10. XAI 분석 - SHAP

> **핵심**: SHAP 분석은 **테스트 데이터** 기준으로 수행합니다.  
> 훈련 데이터로 SHAP을 계산하면 모델이 이미 학습한 데이터에 대한 설명이 되어 실제 예측 해석이 아닙니다.

In [ ]:
# 테스트 데이터 인덱스 초기화
x_test_r = x_test.reset_index(drop=True)
y_test_r = y_test.reset_index(drop=True)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(x_test_r)  # ← 테스트 데이터 사용

print("SHAP values shape:", shap_values.shape)

In [ ]:
# 예측 결과 기반으로 대표 환자 선택
y_pred_test = model.predict(x_test_r)
Person1 = int(np.where(y_pred_test == 1)[0][0])  # 심장질환 예측
Person2 = int(np.where(y_pred_test == 0)[0][0])  # 정상 예측

print(f"Person1: index={Person1}, 실제={y_test_r[Person1]}, 예측={y_pred_test[Person1]}")
print(f"Person2: index={Person2}, 실제={y_test_r[Person2]}, 예측={y_pred_test[Person2]}")

### Force Plot - 개별 환자 해석

- **Force Plot**: 각 특성이 예측값을 기준값(base value) 대비 얼마나 올리거나 내리는지 표현
- 빨간색(→): 심장질환 예측 방향으로 기여
- 파란색(←): 정상 예측 방향으로 기여

In [ ]:
# Person1 - 심장질환 예측 환자
print(f"--- Person1 (index={Person1}) ---")
display(x_test_r.iloc[Person1])
shap.force_plot(explainer.expected_value, shap_values[Person1, :], x_test_r.iloc[Person1, :])

In [ ]:
idx = Person1
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X["ST_Slope_Flat"], y, color="black", alpha=0.5)
axes[0].scatter(x_test_r["ST_Slope_Flat"].iloc[idx], y_test_r[idx], c="red", s=150, zorder=5, label="Person1")
axes[0].set_title("ST_Slope_Flat vs HeartDisease (전체 데이터)")
axes[0].set_xlabel("ST_Slope_Flat"); axes[0].set_ylabel("HeartDisease"); axes[0].legend()

axes[1].scatter(x_test_r["ST_Slope_Flat"], y_test_r, color="black", alpha=0.5)
axes[1].scatter(x_test_r["ST_Slope_Flat"].iloc[idx], y_test_r[idx], c="red", s=150, zorder=5, label="Person1")
axes[1].set_title("ST_Slope_Flat vs HeartDisease (테스트 데이터)")
axes[1].set_xlabel("ST_Slope_Flat"); axes[1].set_ylabel("HeartDisease")
axes[1].set_xlim(-0.25, 1.25); axes[1].set_ylim(-0.25, 1.25); axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Person2 - 정상 예측 환자
print(f"--- Person2 (index={Person2}) ---")
display(x_test_r.iloc[Person2])
shap.force_plot(explainer.expected_value, shap_values[Person2, :], x_test_r.iloc[Person2, :])

In [ ]:
idx2 = Person2
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X["ST_Slope_Flat"], y, color="black", alpha=0.5)
axes[0].scatter(x_test_r["ST_Slope_Flat"].iloc[idx2], y_test_r[idx2], c="royalblue", s=150, zorder=5, label="Person2")
axes[0].set_title("ST_Slope_Flat vs HeartDisease (전체 데이터)")
axes[0].set_xlabel("ST_Slope_Flat"); axes[0].set_ylabel("HeartDisease"); axes[0].legend()

axes[1].scatter(x_test_r["ST_Slope_Flat"], y_test_r, color="black", alpha=0.5)
axes[1].scatter(x_test_r["ST_Slope_Flat"].iloc[idx2], y_test_r[idx2], c="royalblue", s=150, zorder=5, label="Person2")
axes[1].set_title("ST_Slope_Flat vs HeartDisease (테스트 데이터)")
axes[1].set_xlabel("ST_Slope_Flat"); axes[1].set_ylabel("HeartDisease")
axes[1].set_xlim(-0.25, 1.25); axes[1].set_ylim(-0.25, 1.25); axes[1].legend()

plt.tight_layout()
plt.show()

### Summary Plot - 전체 특성 중요도

- **Bar plot**: 각 특성의 평균 |SHAP| 값 → 전체적인 중요도 순위
- **Beeswarm plot**: 특성값(색상)이 예측에 미치는 방향과 크기를 동시에 표현

In [ ]:
# 특성 중요도 (Bar)
shap.summary_plot(shap_values, x_test_r, plot_type="bar")

In [ ]:
# Beeswarm - 특성값과 SHAP값 분포
shap.summary_plot(shap_values, x_test_r)

In [ ]:
# 전체 데이터 Force Plot
# (90도 회전해서 보면 개별 force plot들의 집합)
shap.force_plot(explainer.expected_value, shap_values, x_test_r)

### Dependence Plot - 주요 특성 상세 분석

- x축: 특성값, y축: 해당 특성의 SHAP값
- 특성값이 변함에 따라 예측에 미치는 영향이 어떻게 달라지는지 확인

In [ ]:
shap.dependence_plot("ST_Slope_Flat",      shap_values, x_test_r, interaction_index=None)

In [ ]:
shap.dependence_plot("ChestPainType_ASY", shap_values, x_test_r, interaction_index=None)

In [ ]:
shap.dependence_plot("Oldpeak",            shap_values, x_test_r, interaction_index=None)

In [ ]:
shap.dependence_plot("ExerciseAngina",     shap_values, x_test_r, interaction_index=None)

In [ ]:
shap.dependence_plot("ST_Slope_Up",        shap_values, x_test_r, interaction_index=None)